# 06 - Pruning and early stopping of trials

This notebook confirms the intermediate-value pruning mechanism that
`TrialTrainer._trial_callback` (in `pipelines.tuning_pipeline.trial`) relies on:
each epoch the trainer calls `trial.report(val_loss, epoch)` and raises
`TrialPruned` when `trial.should_prune()` returns true. The pruner is the
`MedianPruner` configured by the orchestrator's `_create_study`.

We reproduce that exact reporting loop with a synthetic per-epoch loss curve so
the pruning decisions are visible and verifiable by eye.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## Synthetic learning curves and the reporting loop

Each trial has a hidden quality. Good trials descend steeply; poor trials
plateau high. The objective mirrors `TrialTrainer`: report each epoch, then
consult `should_prune`. We use the production `MedianPruner` settings.

In [ ]:
from configuration.tuning_config import Phase1TuneConfig

cfg     = Phase1TuneConfig()
N_EPOCH = 30

def synthetic_curve(quality, rng):
    floor = 0.05 + 0.9 * (1.0 - quality)
    decay = 0.15 + 0.35 * quality
    epochs = np.arange(N_EPOCH)
    curve  = floor + (1.0 - floor) * np.exp(-decay * epochs)
    return curve + 0.01 * rng.standard_normal(N_EPOCH)

report_log = []

def objective(trial):
    quality = trial.suggest_float('quality', 0.0, 1.0)
    rng     = np.random.default_rng(trial.number)
    curve   = synthetic_curve(quality, rng)
    for epoch, val_loss in enumerate(curve):
        trial.report(float(val_loss), epoch)
        if trial.should_prune():
            report_log.append((trial.number, epoch, list(curve[:epoch + 1]), 'PRUNED'))
            raise optuna.exceptions.TrialPruned()
    report_log.append((trial.number, N_EPOCH - 1, list(curve), 'COMPLETE'))
    return float(curve[-1])

pruner = optuna.pruners.MedianPruner(
    n_startup_trials = cfg.pruner_n_startup_trials,
    n_warmup_steps   = cfg.pruner_n_warmup_steps,
)
study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=42),
    pruner    = pruner,
)
study.optimize(objective, n_trials=40)

states = [t.state.name for t in study.trials]
from collections import Counter
print('trial states:', dict(Counter(states)))

## Reported curves with pruning points marked

Pruned trials terminate early (a marker at the last reported epoch); complete
trials run all epochs. Pruned curves should be the ones tracking above the
running median.

In [ ]:
fig, ax = plt.subplots()
for number, last_epoch, curve, state in report_log:
    if state == 'PRUNED':
        ax.plot(curve, color='indianred', alpha=0.5, lw=1)
        ax.plot(last_epoch, curve[-1], 'x', color='red', markersize=7)
    else:
        ax.plot(curve, color='steelblue', alpha=0.5, lw=1)

from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], color='steelblue', label='complete'),
    Line2D([0], [0], color='indianred', label='pruned'),
    Line2D([0], [0], marker='x', color='red', ls='', label='prune point'),
]
ax.legend(handles=handles)
ax.set_xlabel('epoch')
ax.set_ylabel('reported val_loss')
ax.set_title('Intermediate-value pruning (MedianPruner)')
fig.tight_layout()
plt.show()

## Where trials were stopped

Histogram of the epoch at which each trial finished. Pruned trials stop after
the warmup window; complete trials reach the final epoch.

In [ ]:
pruned_epochs   = [le for _, le, _, s in report_log if s == 'PRUNED']
complete_epochs = [le for _, le, _, s in report_log if s == 'COMPLETE']

fig, ax = plt.subplots()
bins = np.arange(0, N_EPOCH + 2) - 0.5
ax.hist([complete_epochs, pruned_epochs], bins=bins, stacked=True,
        color=['steelblue', 'indianred'], label=['complete', 'pruned'])
ax.axvline(cfg.pruner_n_warmup_steps - 0.5, color='k', ls='--', lw=1, label='warmup steps')
ax.set_xlabel('final reported epoch')
ax.set_ylabel('number of trials')
ax.set_title('Stopping epoch by outcome')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

A non-zero count of pruned trials appears in the state summary. In the curve
plot, red (pruned) curves stop early with an x marker while blue curves run the
full thirty epochs. The stopping histogram shows pruned trials concentrated
shortly after the warmup window and complete trials at the final epoch.